# Phase 3 — Exploratory Data Analysis

Financial Fraud Detection & Risk Analytics System

This notebook analyses `data/raw/financial_fraud_detection_dataset.csv` only.

**Scope:** EDA. No model training, SMOTE, scaling, encoding, risk scores, or Streamlit pages.

Statistics are calculated from the loaded DataFrame. The raw CSV is never overwritten.


In [1]:
from pathlib import Path
import sys

from IPython.display import display

start = Path.cwd().resolve()
project_root = None
for candidate in [start, *start.parents]:
    if (
        (candidate / "src").is_dir()
        and (candidate / "data").is_dir()
        and (candidate / "run_validation.py").exists()
    ):
        project_root = candidate
        break
if project_root is None:
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.eda import (
    add_temporary_eda_features,
    amount_bin_summary,
    build_eda_summary_markdown,
    categorical_fraud_summary,
    correlation_matrix,
    customer_summary,
    ensure_eda_artifact_dir,
    international_summary,
    keyword_international_summary,
    numeric_by_target,
    overview_frame,
    raw_dataset_md5,
    target_summary,
    write_eda_summary,
)
from src.data.loader import load_fraud_dataset, validate_fraud_dataset
from src.utils.paths import EDA_ARTIFACTS_DIR, RAW_DATASET_PATH
from src.utils.seeds import RANDOM_STATE, set_global_seed

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
set_global_seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("PROJECT_ROOT resolved via src.utils.paths after sys.path insert")
print("RANDOM_STATE =", RANDOM_STATE)
print("Dataset path (project-relative):", RAW_DATASET_PATH.relative_to(project_root).as_posix())


PROJECT_ROOT resolved via src.utils.paths after sys.path insert
RANDOM_STATE = 42
Dataset path (project-relative): data/raw/financial_fraud_detection_dataset.csv


In [2]:
raw_hash_before = raw_dataset_md5()
df = load_fraud_dataset()
report = validate_fraud_dataset(df, source_path=RAW_DATASET_PATH)
assert report.passed, report.errors
print(report.profile["rows"], "rows,", report.profile["columns"], "columns")
print("Validation passed:", report.passed)
print("Raw MD5 before EDA:", raw_hash_before)


5000 rows, 14 columns
Validation passed: True
Raw MD5 before EDA: 9a4a90ce2e07a717b4289dc95a71663c


## 1. Dataset overview

What does the dataset look like? Shape, dtypes, unique counts, missing values, and duplicates are calculated below.


In [3]:
overview = overview_frame(df)
tgt = target_summary(df)
display(pd.DataFrame({
    "metric": ["rows", "columns", "missing_values", "duplicate_rows", "duplicate_transaction_ids", "memory_kb"],
    "value": [
        tgt["rows"],
        tgt["columns"],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        int(df["Transaction_ID"].duplicated().sum()),
        round(df.memory_usage(deep=True).sum() / 1024, 1),
    ],
}))
display(overview)
print("Column names:", list(df.columns))


,metric,value
0,rows,"5,000.0000"
1,columns,14.0000
2,missing_values,0.0000
3,duplicate_rows,0.0000
4,duplicate_transaction_ids,0.0000
5,memory_kb,"2,446.2000"


,column,dtype,unique_values,missing_values
0,Transaction_ID,str,5000,0
1,Customer_ID,str,3847,0
2,Transaction_Date,str,4980,0
3,Transaction_Amount,float64,4302,0
4,Merchant_Category,str,8,0
5,Payment_Method,str,5,0
6,Device_Type,str,3,0
7,Location,str,7,0
8,Is_International,int64,2,0
9,Previous_Transactions,int64,199,0


Column names: ['Transaction_ID', 'Customer_ID', 'Transaction_Date', 'Transaction_Amount', 'Merchant_Category', 'Payment_Method', 'Device_Type', 'Location', 'Is_International', 'Previous_Transactions', 'Average_Spend', 'Account_Age_Days', 'Suspicious_Keyword', 'Fraudulent']


## 2. Target analysis

`Fraudulent = 1` is the minority class. A constant "all legitimate" predictor would already look accurate, so later model selection must not use accuracy alone.


In [4]:
print(f"Fraud count: {tgt['fraud_count']}")
print(f"Legitimate count: {tgt['legitimate_count']}")
print(f"Fraud rate: {tgt['fraud_rate'] * 100:.2f}%")
print(f"Legitimate rate: {tgt['legitimate_rate'] * 100:.2f}%")

target_plot = df.copy()
target_plot["class_label"] = target_plot["Fraudulent"].map({0: "Legitimate", 1: "Fraudulent"})
fig_target = px.bar(
    target_plot["class_label"].value_counts().rename_axis("class_label").reset_index(name="count"),
    x="class_label",
    y="count",
    title="Fraud vs legitimate transactions",
    labels={"class_label": "Class", "count": "Number of transactions"},
    text="count",
    color="class_label",
    color_discrete_map={"Legitimate": "#4C78A8", "Fraudulent": "#E45756"},
)
fig_target.update_traces(textposition="outside", showlegend=False)
fig_target.update_layout(yaxis_title="Number of transactions")
display(fig_target)


Fraud count: 482
Legitimate count: 4518
Fraud rate: 9.64%
Legitimate rate: 90.36%


## 3. Numerical feature analysis

Features: `Transaction_Amount`, `Previous_Transactions`, `Average_Spend`, `Account_Age_Days`.

Extreme values are retained. Fraud can occur in the tail, so outliers are not removed.


In [5]:
num_table = numeric_by_target(df)
display(num_table)

eda_df = add_temporary_eda_features(df)
print("Temporary columns exist only on the copy:", [c for c in eda_df.columns if c not in df.columns])
print("Original frame columns unchanged:", list(df.columns)[:5], "...")

for column in ["Transaction_Amount", "Previous_Transactions", "Average_Spend", "Account_Age_Days"]:
    fig = px.box(
        eda_df.assign(class_label=eda_df["Fraudulent"].map({0: "Legitimate", 1: "Fraudulent"})),
        x="class_label",
        y=column,
        color="class_label",
        title=f"{column} by fraud status",
        labels={"class_label": "Class", column: column.replace("_", " ")},
        color_discrete_map={"Legitimate": "#4C78A8", "Fraudulent": "#E45756"},
        points="outliers",
    )
    fig.update_layout(showlegend=False)
    display(fig)


,feature,Fraudulent,count,mean,std,min,q25,median,q75,max,corr_with_target
0,Transaction_Amount,0,"4,518.0000",78.8806,77.7764,0.0000,22.6350,55.4550,110.3725,595.3400,0.0116
1,Transaction_Amount,1,482.0000,81.9937,89.3738,0.0000,21.4900,55.4300,109.2900,653.8000,0.0116
2,Previous_Transactions,0,"4,518.0000",99.6868,56.9615,1.0000,50.0000,100.0000,148.0000,199.0000,-0.0146
3,Previous_Transactions,1,482.0000,96.8568,57.5927,1.0000,44.0000,96.0000,148.0000,199.0000,-0.0146
4,Average_Spend,0,"4,518.0000",258.3261,139.9142,10.0100,138.0725,261.6400,378.0850,499.9900,0.0011
5,Average_Spend,1,482.0000,258.8552,143.3096,10.4200,146.5150,264.1150,382.1975,499.6400,0.0011
6,Account_Age_Days,0,"4,518.0000","1,009.5680",568.4707,30.0000,529.0000,"1,009.0000","1,496.0000","1,999.0000",0.0086
7,Account_Age_Days,1,482.0000,"1,026.0643",570.5368,38.0000,565.5000,"1,054.0000","1,476.7500","1,996.0000",0.0086


Temporary columns exist only on the copy: ['hour', 'day', 'day_of_week', 'month', 'year', 'year_month', '_parsed_date', 'amount_to_average_ratio']
Original frame columns unchanged: ['Transaction_ID', 'Customer_ID', 'Transaction_Date', 'Transaction_Amount', 'Merchant_Category'] ...


## 4. Transaction amount analysis

Do not convert amount patterns into a hard-coded rule such as `amount > 200000 = fraud`. The observed maximum in this file is far below that threshold.


In [6]:
amt_view = eda_df.assign(class_label=eda_df["Fraudulent"].map({0: "Legitimate", 1: "Fraudulent"}))
fig_amount = px.histogram(
    amt_view,
    x="Transaction_Amount",
    color="class_label",
    barmode="overlay",
    nbins=40,
    title="Transaction amount distribution by class",
    labels={"Transaction_Amount": "Transaction amount", "class_label": "Class", "count": "Transactions"},
    color_discrete_map={"Legitimate": "#4C78A8", "Fraudulent": "#E45756"},
    opacity=0.7,
)
fig_amount.update_layout(yaxis_title="Number of transactions")
display(fig_amount)

bin_table = amount_bin_summary(df)
bin_table["fraud_rate_pct"] = bin_table["fraud_rate"] * 100
display(bin_table)
fig_bins = px.bar(
    bin_table.astype({"amount_bin": str}),
    x="amount_bin",
    y="fraud_rate",
    text=bin_table["fraud_rate"].map(lambda v: f"{v*100:.1f}%"),
    hover_data=["transaction_count", "fraud_count"],
    title="Fraud rate by transaction amount bin",
    labels={"amount_bin": "Amount bin", "fraud_rate": "Fraud rate"},
)
fig_bins.update_traces(textposition="outside")
fig_bins.update_layout(yaxis_tickformat=".0%")
display(fig_bins)
print("Largest amount bin has a small sample; rates there are noisy and are not a decision rule.")


,amount_bin,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
0,"[0, 25)",1360,139,0.1022,10.2206
1,"[25, 50)",989,85,0.0859,8.5945
2,"[50, 100)",1243,126,0.1014,10.1368
3,"[100, 200)",1006,90,0.0895,8.9463
4,"[200, 400)",372,36,0.0968,9.6774
5,"[400, 1000)",30,6,0.2000,20.0000


Largest amount bin has a small sample; rates there are noisy and are not a decision rule.


## 5. International transaction analysis

`Is_International` is associated with fraud in this sample. This is an observed rate difference, not a causal claim.


In [7]:
intl = international_summary(df)
display(intl)
fig_intl = px.bar(
    intl,
    x="segment",
    y="fraud_rate",
    text=intl["fraud_rate"].map(lambda v: f"{v*100:.2f}%"),
    hover_data=["transaction_count", "fraud_count"],
    title="Fraud rate by international status",
    labels={"segment": "Transaction origin", "fraud_rate": "Fraud rate"},
)
fig_intl.update_traces(textposition="outside")
fig_intl.update_layout(yaxis_tickformat=".0%")
display(fig_intl)
print("Crosstab of Is_International x Fraudulent:")
display(pd.crosstab(df["Is_International"], df["Fraudulent"], margins=True))


,Is_International,segment,transaction_count,fraud_count,fraud_rate,average_amount
0,0,Domestic,4559,319,0.0700,79.2943
1,1,International,441,163,0.3696,78.0059


Crosstab of Is_International x Fraudulent:


Fraudulent,0,1,All
Is_International,,,
0,4240,319,4559
1,278,163,441
All,4518,482,5000


## 6–9. Merchant, payment, device, and location

Rates are shown with transaction counts so small-sample categories are not over-read. Location results are fraud rates by location, not evidence that a city causes fraud.


In [8]:
cat_summary = categorical_fraud_summary(df)
display(cat_summary)

def rate_chart(frame, feature, title, filename_stem):
    subset = frame[frame["feature"] == feature].sort_values("fraud_rate", ascending=False)
    fig = px.bar(
        subset,
        x="category",
        y="fraud_rate",
        text=subset["fraud_rate"].map(lambda v: f"{v*100:.1f}%"),
        hover_data=["transaction_count", "fraud_count", "average_amount"],
        title=title,
        labels={"category": feature.replace("_", " "), "fraud_rate": "Fraud rate"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(yaxis_tickformat=".0%")
    return fig

fig_merchant = rate_chart(cat_summary, "Merchant_Category", "Fraud rate by merchant category", "merchant")
fig_payment = rate_chart(cat_summary, "Payment_Method", "Fraud rate by payment method", "payment")
fig_device = rate_chart(cat_summary, "Device_Type", "Fraud rate by device type", "device")
fig_location = rate_chart(cat_summary, "Location", "Fraud rate by location", "location")
display(fig_merchant)
display(fig_payment)
display(fig_device)
display(fig_location)


,feature,category,transaction_count,fraud_count,fraud_rate,average_amount
0,Merchant_Category,Electronics,650,49,0.0754,78.2724
1,Merchant_Category,Entertainment,630,72,0.1143,79.3380
2,Merchant_Category,Fashion,608,62,0.1020,78.0699
3,Merchant_Category,Food,684,74,0.1082,77.1918
4,Merchant_Category,Grocery,629,58,0.0922,81.4020
5,Merchant_Category,Health,606,61,0.1007,81.9672
6,Merchant_Category,Travel,599,56,0.0935,78.3380
7,Merchant_Category,Utilities,594,50,0.0842,79.0896
8,Payment_Method,Credit Card,972,96,0.0988,76.9819
9,Payment_Method,Debit Card,1008,103,0.1022,79.6518


## 10. Date / time analysis

Temporary fields: hour, day, day_of_week, month, year. They are not written to the raw CSV.

The date window is limited (early 2023 through February 2024, and February 2024 is incomplete). Seasonality is not claimed.


In [9]:
print("Parsed min:", eda_df["_parsed_date"].min())
print("Parsed max:", eda_df["_parsed_date"].max())
print("Year counts:")
display(eda_df["year"].value_counts().sort_index())

monthly = (
    eda_df.groupby("year_month")["Fraudulent"]
    .agg(transaction_count="count", fraud_count="sum")
    .assign(fraud_rate=lambda x: x["fraud_count"] / x["transaction_count"])
    .reset_index()
)
fig_month = px.line(
    monthly,
    x="year_month",
    y="fraud_rate",
    markers=True,
    title="Fraud rate by month",
    labels={"year_month": "Month", "fraud_rate": "Fraud rate"},
)
fig_month.update_layout(yaxis_tickformat=".0%")
display(fig_month)

hourly = (
    eda_df.groupby("hour")["Fraudulent"]
    .agg(transaction_count="count", fraud_count="sum")
    .assign(fraud_rate=lambda x: x["fraud_count"] / x["transaction_count"])
    .reset_index()
)
fig_hour = px.bar(
    hourly,
    x="hour",
    y="fraud_rate",
    hover_data=["transaction_count", "fraud_count"],
    title="Fraud rate by hour of day",
    labels={"hour": "Hour of day (0-23)", "fraud_rate": "Fraud rate"},
)
fig_hour.update_layout(yaxis_tickformat=".0%", xaxis=dict(dtick=1))
display(fig_hour)

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily = (
    eda_df.groupby("day_of_week")["Fraudulent"]
    .agg(transaction_count="count", fraud_count="sum")
    .assign(fraud_rate=lambda x: x["fraud_count"] / x["transaction_count"])
    .reindex(dow_order)
    .reset_index()
)
fig_dow = px.bar(
    daily,
    x="day_of_week",
    y="fraud_rate",
    hover_data=["transaction_count", "fraud_count"],
    title="Fraud rate by day of week",
    labels={"day_of_week": "Day of week", "fraud_rate": "Fraud rate"},
)
fig_dow.update_layout(yaxis_tickformat=".0%")
display(fig_dow)


Parsed min: 2023-01-01 02:16:00
Parsed max: 2024-02-21 15:34:00
Year counts:


year
2023    4380
2024     620
Name: count, dtype: int64

## 11. Previous transactions, average spend, and account age

Relationships are measured, not assumed. Account-age groups are EDA-only.


In [10]:
display(eda_df.groupby("Fraudulent")[["Previous_Transactions", "Average_Spend", "Account_Age_Days", "amount_to_average_ratio"]].describe().T)

fig_ratio = px.box(
    amt_view.assign(amount_to_average_ratio=eda_df["amount_to_average_ratio"]),
    x="class_label",
    y="amount_to_average_ratio",
    color="class_label",
    title="Amount-to-average-spend ratio by class (EDA-only feature)",
    labels={"class_label": "Class", "amount_to_average_ratio": "Transaction amount / average spend"},
    color_discrete_map={"Legitimate": "#4C78A8", "Fraudulent": "#E45756"},
    points="outliers",
)
fig_ratio.update_layout(showlegend=False)
display(fig_ratio)

age_bins = pd.cut(df["Account_Age_Days"], bins=[0, 180, 365, 730, 1460, 10000], right=False)
age_table = (
    df.groupby(age_bins, observed=False)["Fraudulent"]
    .agg(transaction_count="count", fraud_count="sum")
    .assign(fraud_rate=lambda x: x["fraud_count"] / x["transaction_count"])
    .reset_index()
    .rename(columns={"Account_Age_Days": "account_age_bin"})
)
display(age_table)
print("Zero Average_Spend rows:", int((df["Average_Spend"] == 0).sum()))
print("amount_to_average_ratio is a Phase 4 candidate. It is not saved into the raw dataset.")


Fraudulent                             0          1
Previous_Transactions   count 4,518.0000   482.0000
                        mean     99.6868    96.8568
                        std      56.9615    57.5927
                        min       1.0000     1.0000
                        25%      50.0000    44.0000
                        50%     100.0000    96.0000
                        75%     148.0000   148.0000
                        max     199.0000   199.0000
Average_Spend           count 4,518.0000   482.0000
                        mean    258.3261   258.8552
                        std     139.9142   143.3096
                        min      10.0100    10.4200
                        25%     138.0725   146.5150
                        50%     261.6400   264.1150
                        75%     378.0850   382.1975
                        max     499.9900   499.6400
Account_Age_Days        count 4,518.0000   482.0000
                        mean  1,009.5680 1,026.0643
                        std     568.4707   570.5368
                        min      30.0000    38.0000
                        25%     529.0000   565.5000
                        50%   1,009.0000 1,054.0000
                        75%   1,496.0000 1,476.7500
                        max   1,999.0000 1,996.0000
amount_to_average_ratio count 4,518.0000   482.0000
                        mean      0.5787     0.6850
                        std       1.2439     1.6257
                        min       0.0000     0.0000
                        25%       0.0905     0.0932
                        50%       0.2400     0.2474
                        75%       0.5767     0.5724
                        max      28.3633    20.0741

,account_age_bin,transaction_count,fraud_count,fraud_rate
0,"[0, 180)",405,42,0.1037
1,"[180, 365)",472,41,0.0869
2,"[365, 730)",909,84,0.0924
3,"[730, 1460)",1861,181,0.0973
4,"[1460, 10000)",1353,134,0.0990


Zero Average_Spend rows: 0
amount_to_average_ratio is a Phase 4 candidate. It is not saved into the raw dataset.


## 12. Suspicious_Keyword leakage analysis

**Suspicious_Keyword requires a provenance/timing review before inclusion in the production model.**

It is strongly associated with the target in this file. That is not proof of leakage, because the dataset documentation does not state when the flag is assigned. It is **not** used as a production ML feature in this phase.


In [11]:
kw = cat_summary[cat_summary["feature"] == "Suspicious_Keyword"]
display(kw)
kw_intl = keyword_international_summary(df)
display(kw_intl)

fig_kw = px.bar(
    kw_intl,
    x="segment",
    y="fraud_rate",
    color="Suspicious_Keyword",
    barmode="group",
    text=kw_intl["fraud_rate"].map(lambda v: f"{v*100:.1f}%"),
    hover_data=["transaction_count", "fraud_count"],
    title="Fraud rate by international status and Suspicious_Keyword",
    labels={"segment": "Origin", "fraud_rate": "Fraud rate", "Suspicious_Keyword": "Suspicious keyword"},
)
fig_kw.update_traces(textposition="outside")
fig_kw.update_layout(yaxis_tickformat=".0%")
display(fig_kw)


,feature,category,transaction_count,fraud_count,fraud_rate,average_amount
25,Suspicious_Keyword,No,4740,359,0.0757,79.0278
26,Suspicious_Keyword,Yes,260,123,0.4731,81.9680


,Is_International,Suspicious_Keyword,transaction_count,fraud_count,fraud_rate,segment
0,0,No,4321,215,0.0498,Domestic
1,0,Yes,238,104,0.4370,Domestic
2,1,No,419,144,0.3437,International
3,1,Yes,22,19,0.8636,International


## 13. Customer_ID analysis

Do not one-hot encode `Customer_ID`. Any later customer aggregate must be computed from training-time history only (Phase 4 consideration).


In [12]:
cust = customer_summary(df)
display(pd.DataFrame({"metric": list(cust), "value": list(cust.values())}))
tx_per_customer = df["Customer_ID"].value_counts()
fig_cust = px.histogram(
    tx_per_customer.rename("transactions_per_customer").reset_index(),
    x="transactions_per_customer",
    title="Transactions per customer",
    labels={"transactions_per_customer": "Transactions per customer", "count": "Number of customers"},
    nbins=5,
)
fig_cust.update_layout(yaxis_title="Number of customers")
display(fig_cust)


,metric,value
0,unique_customers,"3,847.0000"
1,mean_transactions_per_customer,1.2997
2,median_transactions_per_customer,1.0000
3,max_transactions_per_customer,5.0000
4,min_transactions_per_customer,1.0000
5,customers_with_multiple_transactions,952.0000
6,customers_with_any_fraud,472.0000


## 14. Correlation analysis

Pearson correlation does not capture hour-of-day or most categorical associations. It is not used as the sole feature-selection method. Correlation is not causation.


In [13]:
corr = correlation_matrix(df)
display(corr)
fig_corr = go.Figure(
    data=go.Heatmap(
        z=corr.values,
        x=list(corr.columns),
        y=list(corr.index),
        colorscale="RdBu",
        zmid=0,
        colorbar=dict(title="Correlation"),
        text=corr.round(3).astype(str).values,
        texttemplate="%{text}",
        hovertemplate="%{y} vs %{x}: %{z:.3f}<extra></extra>",
    )
)
fig_corr.update_layout(
    title="Correlation matrix for numeric fields, Is_International, and Fraudulent",
    xaxis_title="Feature",
    yaxis_title="Feature",
    width=720,
    height=560,
)
display(fig_corr)


,Transaction_Amount,Previous_Transactions,Average_Spend,Account_Age_Days,Is_International,Fraudulent
Transaction_Amount,1.0000,-0.0114,0.0254,0.0033,-0.0046,0.0116
Previous_Transactions,-0.0114,1.0000,-0.0169,-0.0007,-0.0063,-0.0146
Average_Spend,0.0254,-0.0169,1.0000,-0.0035,0.0197,0.0011
Account_Age_Days,0.0033,-0.0007,-0.0035,1.0000,0.0056,0.0086
Is_International,-0.0046,-0.0063,0.0197,0.0056,1.0000,0.2879
Fraudulent,0.0116,-0.0146,0.0011,0.0086,0.2879,1.0000


## 15. Categorical fraud-rate summary

Reusable table for later dashboard work: feature, category, counts, fraud rate, average amount.


In [14]:
display(cat_summary)


,feature,category,transaction_count,fraud_count,fraud_rate,average_amount
0,Merchant_Category,Electronics,650,49,0.0754,78.2724
1,Merchant_Category,Entertainment,630,72,0.1143,79.3380
2,Merchant_Category,Fashion,608,62,0.1020,78.0699
3,Merchant_Category,Food,684,74,0.1082,77.1918
4,Merchant_Category,Grocery,629,58,0.0922,81.4020
5,Merchant_Category,Health,606,61,0.1007,81.9672
6,Merchant_Category,Travel,599,56,0.0935,78.3380
7,Merchant_Category,Utilities,594,50,0.0842,79.0896
8,Payment_Method,Credit Card,972,96,0.0988,76.9819
9,Payment_Method,Debit Card,1008,103,0.1022,79.6518


## 16. Candidate features for Phase 4

These labels are EDA recommendations. They are not permanently enforced yet.

| Decision | Variables |
|---|---|
| KEEP | Transaction_Amount, Previous_Transactions, Average_Spend, Account_Age_Days, Is_International, Merchant_Category, Payment_Method, Device_Type, Location, Transaction_Date-derived features |
| TRANSFORM | amount_to_average_ratio, hour, day of week, month |
| EXCLUDE | Transaction_ID, raw Customer_ID |
| INVESTIGATE | Suspicious_Keyword |

`Fraudulent` is the target and must never appear in model features.


## 17. Leakage check

| Column | Potential leakage? | Reason | Action |
|---|---|---|---|
| Fraudulent | Yes (target) | Label to predict | Exclude from features |
| Transaction_ID | Identifier | Unique per row; no generalisation | Exclude |
| Customer_ID | Possible if misused | High cardinality; future aggregates can leak | Exclude raw ID; train-only aggregates later |
| Suspicious_Keyword | Investigate | Strong target association; timing undocumented | Provenance review; not a production feature in this phase |
| Transaction_Date | No, if derived from event time | Known at scoring time if the timestamp is the event time | Derive hour/day/month on a copy |
| Is_International | Unlikely | Transaction attribute with a strong observed association | KEEP |
| Remaining amount/history/category fields | Unlikely | Appear to be transaction attributes | KEEP / TRANSFORM |


## 18. Key findings

Findings below are printed from the same DataFrame used throughout this notebook.


In [15]:
print(build_eda_summary_markdown(df).split("## 11. Key findings")[-1])




1. The dataset has 5000 transactions and 482 fraud cases (9.64% fraud rate), with no missing values and no duplicate Transaction_IDs.
2. International transactions have a higher observed fraud rate (36.96%) than domestic transactions (7.00%).
3. Suspicious_Keyword = Yes is strongly associated with the target (47.31% vs 7.57% when No) and therefore requires provenance review.
4. Transaction amount alone shows limited separation: fraud and legitimate medians are 55.43 and 55.45.
5. Overnight hours (00:00-05:00) have higher observed fraud rates than daytime hours; hour is a useful derived feature candidate.
6. Merchant, payment, device, and location fraud rates vary modestly (roughly 8.58% to 11.43%) with comparable sample sizes.
7. Previous_Transactions, Average_Spend, and Account_Age_Days have near-zero correlations with the target (|r| < 0.02) but remain KEEP candidates for non-linear models.
8. The amount-to-average-spend ratio is slightly higher on average for fraud (0.685 vs 0.579

## 19. Save EDA artifacts

HTML charts and the markdown report are written under project-relative paths. The raw CSV is hashed again to confirm it is unchanged.


In [16]:
artifact_dir = ensure_eda_artifact_dir()

def save_fig(fig, name):
    path = artifact_dir / name
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    print("wrote", path.relative_to(project_root).as_posix())

save_fig(fig_target, "fraud_distribution.html")
save_fig(fig_merchant, "fraud_rate_by_merchant.html")
save_fig(fig_payment, "fraud_rate_by_payment.html")
save_fig(fig_location, "fraud_rate_by_location.html")
save_fig(fig_device, "fraud_rate_by_device.html")
save_fig(fig_intl, "fraud_rate_international.html")
save_fig(fig_hour, "fraud_rate_by_hour.html")
save_fig(fig_amount, "amount_distribution.html")
save_fig(fig_corr, "correlation_matrix.html")

cat_path = artifact_dir / "categorical_fraud_summary.csv"
cat_summary.to_csv(cat_path, index=False)
print("wrote", cat_path.relative_to(project_root).as_posix())

report_path = write_eda_summary(df)
print("wrote", report_path.relative_to(project_root).as_posix())

raw_hash_after = raw_dataset_md5()
assert raw_hash_before == raw_hash_after, "Raw dataset changed during EDA"
print("Raw MD5 after EDA:", raw_hash_after)
print("EDA complete. Raw dataset unchanged.")


wrote artifacts/eda/fraud_distribution.html
wrote artifacts/eda/fraud_rate_by_merchant.html
wrote artifacts/eda/fraud_rate_by_payment.html
wrote artifacts/eda/fraud_rate_by_location.html
wrote artifacts/eda/fraud_rate_by_device.html
wrote artifacts/eda/fraud_rate_international.html
wrote artifacts/eda/fraud_rate_by_hour.html
wrote artifacts/eda/amount_distribution.html
wrote artifacts/eda/correlation_matrix.html
wrote artifacts/eda/categorical_fraud_summary.csv
wrote reports/eda_summary.md
Raw MD5 after EDA: 9a4a90ce2e07a717b4289dc95a71663c
EDA complete. Raw dataset unchanged.
